<a href="https://colab.research.google.com/github/scottaudrey/svd-food-recommendation-system/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part A: Load the Ratings

In [2]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

In [3]:
url = "https://github.com/ryannewhouse11/FoodReviewData/releases/latest/download/recipereview.zip"
urllib.request.urlretrieve(url, "/content/recipereview.zip")

with zipfile.ZipFile("/content/recipereview.zip") as z:
  csv_files = [
      name for name in z.namelist()
      if name.lower().endswith(".csv") and not name.startswith("_MACOSX/")
  ]


  with z.open(csv_files[0]) as f:
    ratings = pd.read_csv(
        f,
        sep=",",
        header=0,
        names=["num_records", "recipe_number", "recipe_code", "recipe_name",
               "comment_id", "user_id", "user_name", "user_reputation", "created_at",
               "reply_count", "thumbs_up", "thumbs_down", "stars", "best_score", "text"]
    )
  print(ratings.head())
  print("Number of ratings:", len(ratings))


   num_records  recipe_number  recipe_code         recipe_name  \
0            0              1        14299  Creamy White Chili   
1            1              1        14299  Creamy White Chili   
2            2              1        14299  Creamy White Chili   
3            3              1        14299  Creamy White Chili   
4            4              1        14299  Creamy White Chili   

                                        comment_id         user_id  \
0  sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM  u_9iFLIhMa8QaG   
1  sp_aUSaElGf_14299_c_2FsPC83HtzCsQAtOxlbL6RcaPbY  u_Lu6p25tmE77j   
2  sp_aUSaElGf_14299_c_2FPrSGyTv7PQkZq37j92r9mYGkP  u_s0LwgpZ8Jsqq   
3  sp_aUSaElGf_14299_c_2DzdSIgV9qNiuBaLoZ7JQaartoC  u_fqrybAdYjgjG   
4  sp_aUSaElGf_14299_c_2DtZJuRQYeTFwXBoZRfRhBPEXjI  u_XXWKwVhKZD69   

    user_name  user_reputation  created_at  reply_count  thumbs_up  \
0     Jeri326                1  1665619889            0          0   
1     Mark467               50  1665277687

# Part B: Make an honest train/test split

In [10]:
def split_ratings(ratings, test_frac=0.10, seed=0):
  rng = np.random.default_rng(seed)
  is_test = np.zeros(len(ratings), dtype=bool)

  user_left = ratings.groupby("user_id").size().to_dict()
  item_left = ratings.groupby("recipe_code").size().to_dict()

  for pos in rng.permutation(len(ratings)):  # goes through the ratings randomly
    row = ratings.iloc[pos]
    user_id, recipe_code = row["user_id"], row["recipe_code"]

    if (rng.random() < test_frac   # each of the ratings now has a 10% chance of being the test rating
      and user_left[user_id] > 1
      and item_left[recipe_code] > 1):
      is_test[pos] = True
      user_left[user_id] -= 1
      item_left[recipe_code] -= 1

  return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()  # returns the integer

train, test = split_ratings(ratings)
print("train: ", len(train), "test: ", len(test))
print("test fraction: ", round(len(test) / len(ratings), 3))

train:  17543 test:  639
test fraction:  0.035
